In [ ]:
!pip install web3


In [ ]:
from web3 import Web3
import json

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
contract_abi = [
    {
        "inputs": [
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "grantAccess",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "revokeAccess",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "address", "name": "patient", "type": "address"},
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "checkAccess",
        "outputs": [
            {"internalType": "bool", "name": "", "type": "bool"}
        ],
        "stateMutability": "view",
        "type": "function"
    }
]

print("Contract ABI prepared successfully!")


Contract ABI prepared successfully!


In [ ]:
# Sepolia blockchain connection
rpc_url = "YOUR_SEPOLIA_RPC_URL"

# Connect to Ethereum
w3 = Web3(Web3.HTTPProvider(rpc_url))

print("Connected:", w3.is_connected())


Connected: False


In [ ]:
from google.colab import userdata
from web3 import Web3

rpc_url = userdata.get("ALCHEMY_RPC_URL")

w3 = Web3(Web3.HTTPProvider(rpc_url))

if w3.is_connected():
    print("Successfully connected to Ethereum!")
    print("Connected network chain ID:", w3.eth.chain_id)
else:
    print("Connection failed. Check your RPC URL.")


Successfully connected to Ethereum!
Connected network chain ID: 11155111


In [ ]:
contract_address = "0x592c0909764c166d26DDcbEEBcd57816cA13C2b4"

contract_address = Web3.to_checksum_address(contract_address)

print("Contract address loaded successfully!")


Contract address loaded successfully!


In [ ]:
healthcare_contract = w3.eth.contract(
    address=contract_address,
    abi=contract_abi
)

print("Healthcare contract connected successfully!")


Healthcare contract connected successfully!


In [ ]:
patient_address = Web3.to_checksum_address(
    "0x458bc012d1d59a71a13cc041c001a84bc0e32374"
)

doctor_address = Web3.to_checksum_address(
    "0xAb8483F64d9C6d1EcF9b849Ae677dD3315835cb2"
)

record_id = 1

In [ ]:
has_access = healthcare_contract.functions.checkAccess(
    patient_address,
    doctor_address,
    record_id
).call()

print("Doctor access:", has_access)

Doctor access: True


In [ ]:
from google.colab import userdata
from eth_account import Account

# Retrieve the private key securely from Colab Secrets
private_key = userdata.get("PATIENT_PRIVATE_KEY")

# Load the wallet account
patient_account = Account.from_key(private_key)

# Display the public address only
patient_address = Web3.to_checksum_address(patient_account.address)

print("Patient wallet connected!")
print("Wallet address:", patient_address)
print("Network chain ID:", w3.eth.chain_id)
print("Sepolia ETH balance:", w3.from_wei(
    w3.eth.get_balance(patient_address), "ether"
))

Patient wallet connected!
Wallet address: 0xE0386AeD51BD7fa0A7b1F534cbC7D06682098F56
Network chain ID: 11155111
Sepolia ETH balance: 0.07


In [ ]:
healthcare_contract = w3.eth.contract(
    address=contract_address,
    abi=contract_abi
)

print("Contract connected!")
print("Patient wallet:", patient_address)

Contract connected!
Patient wallet: 0xE0386AeD51BD7fa0A7b1F534cbC7D06682098F56


In [ ]:
doctor_address = Web3.to_checksum_address(
    "0x41506d22a210283196f148bB8164cE0484656443"
)

record_id = 1

print("Doctor address:", doctor_address)
print("Record ID:", record_id)

Doctor address: 0x41506d22a210283196f148bB8164cE0484656443
Record ID: 1


In [ ]:
# Check whether access has already been granted
already_granted = healthcare_contract.functions.checkAccess(
    patient_address,
    doctor_address,
    record_id
).call()

if already_granted:
    print("This doctor already has access to this record.")

else:
    # Prepare the transaction
    nonce = w3.eth.get_transaction_count(patient_address)

    transaction = healthcare_contract.functions.grantAccess(
        doctor_address,
        record_id
    ).build_transaction({
        "from": patient_address,
        "nonce": nonce,
        "chainId": 11155111,
        "gasPrice": w3.eth.gas_price
    })

    # Estimate transaction gas
    gas_estimate = w3.eth.estimate_gas(transaction)
    transaction["gas"] = int(gas_estimate * 1.2)

    # Sign using the patient wallet
    signed_transaction = patient_account.sign_transaction(transaction)

    # Send transaction to Sepolia
    tx_hash = w3.eth.send_raw_transaction(
        signed_transaction.raw_transaction
    )

    print("Transaction sent!")
    print("Transaction hash:", tx_hash.hex())

    # Wait for blockchain confirmation
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    if receipt.status == 1:
        print("SUCCESS! Doctor access granted.")
    else:
        print("Transaction failed.")

Transaction sent!
Transaction hash: d9a38c0724ce02a4f1226c2b0d4aa32be322f109db26f2a98691bcbd23d4bb57
SUCCESS! Doctor access granted.


In [ ]:
has_access = healthcare_contract.functions.checkAccess(
    patient_address,
    doctor_address,
    record_id
).call()

print("Doctor access:", has_access)

Doctor access: True


In [ ]:
# Prepare the revoke transaction
nonce = w3.eth.get_transaction_count(patient_address)

transaction = healthcare_contract.functions.revokeAccess(
    doctor_address,
    record_id
).build_transaction({
    "from": patient_address,
    "nonce": nonce,
    "chainId": 11155111,
    "gasPrice": w3.eth.gas_price
})

# Estimate gas
gas_estimate = w3.eth.estimate_gas(transaction)
transaction["gas"] = int(gas_estimate * 1.2)

# Sign and send
signed_transaction = patient_account.sign_transaction(transaction)

tx_hash = w3.eth.send_raw_transaction(
    signed_transaction.raw_transaction
)

print("Revoke transaction sent:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

if receipt.status == 1:
    print("SUCCESS! Doctor access revoked.")
else:
    print("Transaction failed.")

Revoke transaction sent: 4a8527a53e2d515ff0c03caaf54ab9ab3dbebc1298f894362ad713b2f18fdcdf
SUCCESS! Doctor access revoked.


In [ ]:
has_access = healthcare_contract.functions.checkAccess(
    patient_address,
    doctor_address,
    record_id
).call()

print("Doctor access:", has_access)

Doctor access: False


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 33.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 71.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
import streamlit as st

print("Streamlit installed successfully!")
print("Streamlit version:", st.__version__)

Streamlit installed successfully!
Streamlit version: 1.67.0


In [ ]:
%%writefile app.py

import streamlit as st

st.set_page_config(
    page_title="Healthcare Access Management",
    page_icon="🏥",
    layout="centered"
)

st.title("🏥 Healthcare Access Management")

st.write(
    "A decentralised application that allows patients "
    "to manage healthcare professional access to their records."
)

st.divider()

st.subheader("Manage Healthcare Record Access")

doctor_address = st.text_input(
    "Doctor's Wallet Address",
    placeholder="Enter doctor's wallet address"
)

record_id = st.number_input(
    "Healthcare Record ID",
    min_value=1,
    step=1
)

st.write("### Manage Permissions")

col1, col2 = st.columns(2)

with col1:
    st.button("Grant Access", use_container_width=True)

with col2:
    st.button("Revoke Access", use_container_width=True)

st.divider()

st.subheader("Check Access Permission")

st.button("Check Permission", use_container_width=True)

st.caption(
    "Prototype demonstration only. No real patient information is stored."
)

Writing app.py


In [ ]:
import os

print("App file exists:", os.path.exists("app.py"))

App file exists: True


In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.3
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.3
npm notice To update run: npm install -g npm@12.2.3
npm notice
⠧

In [ ]:
!streamlit run app.py &>/content/streamlit.log &

In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://twelve-icons-study.loca.lt
^C


In [ ]:
import os
from google.colab import userdata

# Load secrets into the environment (never print their values)
os.environ["ALCHEMY_RPC_URL"] = userdata.get("ALCHEMY_RPC_URL")
os.environ["PATIENT_PRIVATE_KEY"] = userdata.get("PATIENT_PRIVATE_KEY")
os.environ["CONTRACT_ADDRESS"] = userdata.get("CONTRACT_ADDRESS")
os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")

print("All application secrets loaded successfully!")

All application secrets loaded successfully!


In [2]:
%%writefile app.py

import os
import streamlit as st
from web3 import Web3
from eth_account import Account

# -----------------------------
# 1. Page setup
# -----------------------------

st.set_page_config(
    page_title="Healthcare Access Management",
    page_icon="🏥",
    layout="centered"
)

st.title("🏥 Healthcare Access Management")
st.write(
    "A decentralised application allowing patients "
    "to manage healthcare professional access to records."
)

# -----------------------------
# 2. Password protection
# -----------------------------

app_password = os.environ.get("APP_PASSWORD")

if not app_password:
    st.error("Application password is not configured.")
    st.stop()

if not st.session_state.get("authenticated", False):
    entered_password = st.text_input(
        "Enter application password",
        type="password"
    )

    if st.button("Log in"):
        if entered_password == app_password:
            st.session_state.authenticated = True
            st.rerun()
        else:
            st.error("Incorrect password.")

    st.stop()

# -----------------------------
# 3. Connect to blockchain
# -----------------------------

try:
    rpc_url = os.environ["ALCHEMY_RPC_URL"]
    private_key = os.environ["PATIENT_PRIVATE_KEY"]
    contract_address = Web3.to_checksum_address(
        os.environ["CONTRACT_ADDRESS"]
    )

    w3 = Web3(Web3.HTTPProvider(rpc_url))

    if not w3.is_connected():
        st.error("Could not connect to Sepolia.")
        st.stop()

    if w3.eth.chain_id != 11155111:
        st.error("Connected network is not Sepolia.")
        st.stop()

    patient_account = Account.from_key(private_key)
    patient_address = Web3.to_checksum_address(
        patient_account.address
    )

except Exception:
    st.error("Blockchain configuration failed. Check your Colab setup.")
    st.stop()

# -----------------------------
# 4. Smart contract ABI
# -----------------------------

contract_abi = [
    {
        "inputs": [
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "grantAccess",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "revokeAccess",
        "outputs": [],
        "stateMutability": "nonpayable",
        "type": "function"
    },
    {
        "inputs": [
            {"internalType": "address", "name": "patient", "type": "address"},
            {"internalType": "address", "name": "doctor", "type": "address"},
            {"internalType": "uint256", "name": "recordId", "type": "uint256"}
        ],
        "name": "checkAccess",
        "outputs": [
            {"internalType": "bool", "name": "", "type": "bool"}
        ],
        "stateMutability": "view",
        "type": "function"
    }
]

contract = w3.eth.contract(
    address=contract_address,
    abi=contract_abi
)

# -----------------------------
# 5. Display patient account
# -----------------------------

st.markdown(
    '<p class="connection-status">● Connected to Sepolia blockchain</p>',
    unsafe_allow_html=True
)

st.write("**Patient wallet:**")
st.code(patient_address)

st.divider()

# -----------------------------
# 6. User inputs
# -----------------------------

st.subheader("Manage Healthcare Record Access")

doctor_input = st.text_input(
    "Doctor's wallet address",
    placeholder="Enter doctor's public wallet address"
)

record_id = st.number_input(
    "Healthcare record ID",
    min_value=1,
    step=1
)

# -----------------------------
# 7. Validate doctor address
# -----------------------------

def get_doctor_address():
    try:
        doctor = Web3.to_checksum_address(doctor_input.strip())

        if doctor == "0x0000000000000000000000000000000000000000":
            st.error("The zero address is not valid.")
            return None

        if doctor == patient_address:
            st.error("You cannot grant access to yourself.")
            return None

        return doctor

    except Exception:
        st.error("Please enter a valid Ethereum wallet address.")
        return None

# -----------------------------
# 8. Send blockchain transaction
# -----------------------------

def send_permission_transaction(function_call):
    try:
        nonce = w3.eth.get_transaction_count(
            patient_address,
            "pending"
        )

        transaction = function_call.build_transaction({
            "from": patient_address,
            "nonce": nonce,
            "chainId": 11155111,
            "gasPrice": w3.eth.gas_price
        })

        gas_estimate = w3.eth.estimate_gas(transaction)
        transaction["gas"] = int(gas_estimate * 1.2)

        signed_transaction = patient_account.sign_transaction(
            transaction
        )

        tx_hash = w3.eth.send_raw_transaction(
            signed_transaction.raw_transaction
        )

        st.info("Transaction submitted. Waiting for confirmation...")

        receipt = w3.eth.wait_for_transaction_receipt(
            tx_hash,
            timeout=120
        )

        st.write("Transaction hash:")
        st.code(tx_hash.hex())

        if receipt.status == 1:
            st.success("Transaction confirmed on Sepolia!")
            st.write("Block number:", receipt.blockNumber)
        else:
            st.error("The blockchain transaction failed.")

    except Exception as error:
        st.error(
            "Transaction could not be completed. "
            "Check the permission status and wallet balance."
        )
        st.caption(str(error)[:500])

# -----------------------------
# 9. Grant and revoke buttons
# -----------------------------

col1, col2 = st.columns(2)

with col1:
    if st.button("Grant Access", use_container_width=True):
        doctor = get_doctor_address()

        if doctor:
            send_permission_transaction(
                contract.functions.grantAccess(
                    doctor,
                    int(record_id)
                )
            )

with col2:
    if st.button("Revoke Access", use_container_width=True):
        doctor = get_doctor_address()

        if doctor:
            send_permission_transaction(
                contract.functions.revokeAccess(
                    doctor,
                    int(record_id)
                )
            )

# -----------------------------
# 10. Check permissions
# -----------------------------

st.divider()
st.subheader("Check Access Permission")

if st.button("Check Permission", use_container_width=True):
    doctor = get_doctor_address()

    if doctor:
        try:
            permission = contract.functions.checkAccess(
                patient_address,
                doctor,
                int(record_id)
            ).call()

            if permission:
                st.success("Doctor currently HAS access.")
            else:
                st.warning("Doctor currently DOES NOT have access.")

        except Exception:
            st.error("Unable to check permission.")

st.divider()

st.caption(
    "Educational prototype using fictional record IDs. "
    "No real patient information is stored."
)

Overwriting app.py


In [ ]:
!pkill -f "streamlit run app.py" || true

^C


In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼your url is: https://witty-cougars-bake.loca.lt
^C


In [ ]:
!cat /content/streamlit.log



2026-11-24 20:43:44.226 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.153.21.161:8501

  Stopping...


In [ ]:
import subprocess
import time
import requests

# Start Streamlit in the background
log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Check whether the application responds
try:
    response = requests.get("http://localhost:8501")
    print("Streamlit status:", response.status_code)
except Exception as error:
    print("Connection problem:", error)

Streamlit status: 200


In [ ]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

old_title = 'st.title("🏥 Healthcare Access Management")'

new_title = '''
st.markdown("""
<style>
    .stApp {
        background-color: #F4F8F7;
    }

    h1, h2, h3 {
        color: #03594c !important;
    }

    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    .stButton > button {
        background-color: #03594c;
        color: white;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white;
        border-color: #047563;
    }

    .stTextInput input, .stNumberInput input {
        border-radius: 8px;
    }

    [data-testid="stAlert"] {
        border-radius: 10px;
    }

    footer {
        visibility: hidden;
    }
</style>
""", unsafe_allow_html=True)

st.title("🏥 Healthcare Access Management")
'''

if old_title in app_code:
    app_code = app_code.replace(old_title, new_title, 1)
    app_path.write_text(app_code)
    print("HSE knock-off style!")
else:
    print("Couldn't find the title automatically. No changes made.")

HSE knock-off style!


In [ ]:
!pkill -f "streamlit run app.py" || true

^C


In [ ]:
import subprocess
import time

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)
print("Dashboard restarted!")

Dashboard restarted!


In [ ]:
import os
from google.colab import userdata

os.environ["ALCHEMY_RPC_URL"] = userdata.get("ALCHEMY_RPC_URL")
os.environ["PATIENT_PRIVATE_KEY"] = userdata.get("PATIENT_PRIVATE_KEY")
os.environ["CONTRACT_ADDRESS"] = userdata.get("CONTRACT_ADDRESS")
os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")

print("Secrets loaded:", all([
    os.environ.get("ALCHEMY_RPC_URL"),
    os.environ.get("PATIENT_PRIVATE_KEY"),
    os.environ.get("CONTRACT_ADDRESS"),
    os.environ.get("APP_PASSWORD")
]))

Secrets loaded: True


In [ ]:
import subprocess
import time

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)

print("Streamlit running:", streamlit_process.poll() is None)

Streamlit running: True


In [1]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

start = app_code.find('<style>')
end = app_code.find('</style>')

if start != -1 and end != -1:
    new_style = """<style>
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    /* Main text */
    p, label, span, li, .stMarkdown {
        color: #26332F;
    }

    /* Headings */
    h1, h2, h3 {
        color: #03594c !important;
    }

    /* Top bar */
    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    /* Metric cards */
    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    /* Input fields */
    .stTextInput input, .stNumberInput input {
        background-color: white;
        color: #26332F;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    /* Buttons */
    .stButton > button {
        background-color: #03594c;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white !important;
        border-color: #047563;
    }

    /* Alerts */
    [data-testid="stAlert"] {
        border-radius: 10px;
    }

    footer {
        visibility: hidden;
    }
    </style>"""

    app_code = app_code[:start] + new_style + app_code[end + len('</style>'):]

# Removed the hospital emoji from the title
app_code = app_code.replace(
    'st.title("🏥 Healthcare Access Management")',
    'st.title("Healthcare Information Access Management")'
)

app_path.write_text(app_code)
print("Updated stuff.")

Updated stuff.


In [3]:
import os
from google.colab import userdata

os.environ["ALCHEMY_RPC_URL"] = userdata.get("ALCHEMY_RPC_URL")
os.environ["PATIENT_PRIVATE_KEY"] = userdata.get("PATIENT_PRIVATE_KEY")
os.environ["CONTRACT_ADDRESS"] = userdata.get("CONTRACT_ADDRESS")
os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")

print("Secrets loaded successfully!")

Secrets loaded successfully!


In [4]:
import subprocess
import time

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)
print("Dashboard running:", streamlit_process.poll() is None)

Dashboard running: True


In [1]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

app_code = app_code.replace(
    "h1, h2, h3 {\n        color: #03594c !important;\n    }",
    """h1, h2, h3 {
        color: #047563 !important;
    }

    /* Blockchain connection status */
    .connection-status {
        color: #047563 !important;
        font-weight: 600;
    }

    /* Section headings */
    h2, h3 {
        color: #168577 !important;
    }"""
)

app_path.write_text(app_code)
print("Colour updates.")

Colour updates.


In [4]:
import subprocess
import time
import os

# Stop the previous Streamlit process
subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)

CompletedProcess(args=['pkill', '-f', 'streamlit run app.py'], returncode=1)

In [5]:
log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)
print("Dashboard running:", streamlit_process.poll() is None)

Dashboard running: True


In [7]:
import os
from google.colab import userdata

os.environ["ALCHEMY_RPC_URL"] = userdata.get("ALCHEMY_RPC_URL")
os.environ["PATIENT_PRIVATE_KEY"] = userdata.get("PATIENT_PRIVATE_KEY")
os.environ["CONTRACT_ADDRESS"] = userdata.get("CONTRACT_ADDRESS")
os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")

print("App password loaded:", bool(os.environ.get("APP_PASSWORD")))
print("All app settings loaded:", all([
    os.environ.get("ALCHEMY_RPC_URL"),
    os.environ.get("PATIENT_PRIVATE_KEY"),
    os.environ.get("CONTRACT_ADDRESS"),
    os.environ.get("APP_PASSWORD")
]))

App password loaded: True
All app settings loaded: True


In [8]:
import subprocess
import time
import os

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)

print("Dashboard running:", streamlit_process.poll() is None)

Dashboard running: True


In [11]:
from pathlib import Path

app_code = Path("app.py").read_text()

print("HSE background colour present:", "#F4F8F7" in app_code)
print("HSE teal present:", "#03594c" in app_code)
print("Updated connection message present:", "connection-status" in app_code)
print("Plain title present:", 'st.title("Healthcare Access Management")' in app_code)

print("\n--- Styling found in app.py ---")
start = app_code.find("<style>")
end = app_code.find("</style>")

if start != -1 and end != -1:
    print(app_code[start:end + 8])
else:
    print("No CSS styling block found.")

HSE background colour present: False
HSE teal present: False
Updated connection message present: True
Plain title present: False

--- Styling found in app.py ---
No CSS styling block found.


In [12]:
from pathlib import Path

app_code = Path("app.py").read_text()

for i, line in enumerate(app_code.splitlines()):
    if "Healthcare Access" in line or "st.title" in line or "st.header" in line:
        print(f"Line {i + 1}: {line}")

Line 12:     page_title="Healthcare Access Management",
Line 17: st.title("🏥 Healthcare Access Management")


In [13]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

# Add the new visual styling immediately before the title
styling = """
st.markdown("""
<style>
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    h1, h2, h3 {
        color: #168577 !important;
    }

    p, label, li {
        color: #26332F;
    }

    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    .stTextInput input, .stNumberInput input {
        background-color: white;
        color: #26332F;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    .stButton > button {
        background-color: #03594c;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white !important;
        border-color: #047563;
    }

    .connection-status {
        color: #168577 !important;
        font-weight: 600;
    }

    footer {
        visibility: hidden;
    }
</style>
""", unsafe_allow_html=True)

"""

title = 'st.title("🏥 Healthcare Access Management")'

if title in app_code:
    app_code = app_code.replace(title, styling + 'st.title("Healthcare Access Management")', 1)
    app_path.write_text(app_code)
    print("HSE-inspired design restored successfully!")
else:
    print("Title not found. No changes made.")

SyntaxError: invalid decimal literal (1883973980.py, line 29)

In [15]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

styling = '''
st.markdown("""
<style>
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    h1, h2, h3 {
        color: #168577 !important;
    }

    p, label, li {
        color: #26332F;
    }

    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    .stTextInput input, .stNumberInput input {
        background-color: white;
        color: #26332F;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    .stButton > button {
        background-color: #03594c;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white !important;
        border-color: #047563;
    }

    .connection-status {
        color: #168577 !important;
        font-weight: 600;
    }

    footer {
        visibility: hidden;
    }
</style>
""", unsafe_allow_html=True)

'''

old_title = 'st.title("🏥 Healthcare Access Management")'
new_title = 'st.title("Healthcare Access Management")'

if old_title in app_code:
    app_code = app_code.replace(old_title, styling + new_title, 1)
    app_path.write_text(app_code)
    print("HSE-inspired design is back 😭")
else:
    print("Couldn't find the old title. No changes made.")

Couldn't find the old title. No changes made.


In [16]:
from pathlib import Path

app_code = Path("app.py").read_text()

for i, line in enumerate(app_code.splitlines()):
    if "st.title" in line:
        print("Exact title line:", repr(line))

Exact title line: 'st.title("Healthcare Access Management")'


In [17]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

styling = '''
st.markdown("""
<style>
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    h1, h2, h3 {
        color: #168577 !important;
    }

    p, label, li {
        color: #26332F;
    }

    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    .stTextInput input, .stNumberInput input {
        background-color: white;
        color: #26332F;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    .stButton > button {
        background-color: #03594c;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white !important;
        border-color: #047563;
    }

    .connection-status {
        color: #168577 !important;
        font-weight: 600;
    }

    footer {
        visibility: hidden;
    }
</style>
""", unsafe_allow_html=True)

'''

title = 'st.title("Healthcare Access Management")'

if "background-color: #F4F8F7" not in app_code and title in app_code:
    app_code = app_code.replace(title, styling + title, 1)
    app_path.write_text(app_code)
    print("HSE-inspired design successfully added!")
else:
    print("Styling already exists or title could not be found.")

Styling already exists or title could not be found.


In [18]:
from pathlib import Path

app_code = Path("app.py").read_text()

print("File length:", len(app_code))
print("Title found:", 'st.title("Healthcare Access Management")' in app_code)
print("Background colour found:", "#F4F8F7" in app_code)
print("Teal colour found:", "#03594c" in app_code)
print("CSS style tag found:", "<style>" in app_code)

print("\n--- First 45 lines of app.py ---")
for i, line in enumerate(app_code.splitlines()[:45], start=1):
    print(f"{i}: {line}")

File length: 8553
Title found: True
Background colour found: True
Teal colour found: True
CSS style tag found: True

--- First 45 lines of app.py ---
1: 
2: import os
3: import streamlit as st
4: from web3 import Web3
5: from eth_account import Account
6: 
7: # -----------------------------
8: # 1. Page setup
9: # -----------------------------
10: 
11: st.set_page_config(
12:     page_title="Healthcare Access Management",
13:     page_icon="🏥",
14:     layout="centered"
15: )
16: 
17: 
18: st.markdown("""
19: <style>
20:     .stApp {
21:         background-color: #F4F8F7;
22:         color: #26332F;
23:     }
24: 
25:     h1, h2, h3 {
26:         color: #168577 !important;
27:     }
28: 
29:     p, label, li {
30:         color: #26332F;
31:     }
32: 
33:     [data-testid="stHeader"] {
34:         background-color: #F4F8F7;
35:     }
36: 
37:     [data-testid="stMetric"] {
38:         background-color: white;
39:         padding: 16px;
40:         border-radius: 12px;
41:         bord

In [19]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

app_code = app_code.replace(
    'page_icon="🏥"',
    'page_icon="💚"'
)

app_path.write_text(app_code)

print("Page icon updated!")

Page icon updated!


In [20]:
app_code = Path("app.py").read_text()
start = app_code.find("<style>")
end = app_code.find("</style>")

print(app_code[start:end + 8])

<style>
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    h1, h2, h3 {
        color: #168577 !important;
    }

    p, label, li {
        color: #26332F;
    }

    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    .stTextInput input, .stNumberInput input {
        background-color: white;
        color: #26332F;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    .stButton > button {
        background-color: #03594c;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563;
        color: white !important;
        border-color: #047563;
    }

    .connection-status {
        color: #168577

In [21]:
from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

start = app_code.find("<style>")
end = app_code.find("</style>")

new_css = """<style>
    /* Overall background and text */
    .stApp {
        background-color: #F4F8F7;
        color: #26332F;
    }

    /* Main title */
    h1 {
        color: #03594c !important;
    }

    /* Section headings */
    h2, h3 {
        color: #168577 !important;
    }

    /* General text */
    .stMarkdown, .stMarkdown p,
    label, [data-testid="stCaptionContainer"] {
        color: #26332F !important;
    }

    /* Blockchain connection status */
    .connection-status {
        color: #168577 !important;
        font-weight: 700;
    }

    /* Metric cards */
    [data-testid="stMetric"] {
        background-color: white;
        padding: 16px;
        border-radius: 12px;
        border: 1px solid #DCE8E5;
    }

    /* Input fields */
    .stTextInput input, .stNumberInput input {
        background-color: white !important;
        color: #26332F !important;
        border: 1px solid #AABDB7;
        border-radius: 8px;
    }

    /* Buttons */
    .stButton > button {
        background-color: #03594c !important;
        color: white !important;
        border: 1px solid #03594c;
        border-radius: 9px;
        font-weight: 600;
        min-height: 45px;
    }

    .stButton > button:hover {
        background-color: #047563 !important;
        color: white !important;
    }

    /* Page header */
    [data-testid="stHeader"] {
        background-color: #F4F8F7;
    }

    footer {
        visibility: hidden;
    }
</style>"""

if start != -1 and end != -1:
    app_code = app_code[:start] + new_css + app_code[end + len("</style>"):]
    app_path.write_text(app_code)
    print("Improved styling successfully saved!")
else:
    print("CSS block not found. No changes made.")

Improved styling successfully saved!


In [22]:
import os
from google.colab import userdata

os.environ["ALCHEMY_RPC_URL"] = userdata.get("ALCHEMY_RPC_URL")
os.environ["PATIENT_PRIVATE_KEY"] = userdata.get("PATIENT_PRIVATE_KEY")
os.environ["CONTRACT_ADDRESS"] = userdata.get("CONTRACT_ADDRESS")
os.environ["APP_PASSWORD"] = userdata.get("APP_PASSWORD")

print("Secrets loaded:", all([
    os.environ.get("ALCHEMY_RPC_URL"),
    os.environ.get("PATIENT_PRIVATE_KEY"),
    os.environ.get("CONTRACT_ADDRESS"),
    os.environ.get("APP_PASSWORD")
]))

Secrets loaded: True


In [24]:
import subprocess

subprocess.run("pkill -f 'streamlit run app.py'", shell=True)

CompletedProcess(args="pkill -f 'streamlit run app.py'", returncode=-15)

In [25]:
import subprocess
import time
import os

log_file = open("/content/streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(5)

print("Dashboard running:", streamlit_process.poll() is None)

Dashboard running: True


In [ ]:
!npx localtunnel --port 8501

⠙⠹⠸⠼your url is: https://big-cooks-smoke.loca.lt
